## Step 1: Build API pipeline

In [21]:
import shopify
import pandas as pd
from datetime import datetime
import time
import os

In [27]:
token = os.getenv("my_token")
merchant = os.getenv("shopify_merchant")

In [4]:
api_session = shopify.Session(merchant,'2026-01', token)
shopify.ShopifyResource.activate_session(api_session)

In [5]:
def get_data(object_name):
    all_data = []
    attribute = getattr(shopify, object_name)
    
    print(f'Fetching first page of {object_name}...')
    data = attribute.find(since_id=0, limit=250, status='any')
    
    for d in data:
        all_data.append(d)
    
    page_count = 1
    
    while data.has_next_page():
        time.sleep(0.5)
        page_count += 1
        print(f"Fetching page {page_count}... Total records collected: {len(all_data)}")
        data = data.next_page()
        for d in data:
            all_data.append(d)
            
    print("Done!")
    return all_data

In [6]:
orders = get_data('Order')

Fetching first page of Order...
Fetching page 2... Total records collected: 250
Fetching page 3... Total records collected: 500
Fetching page 4... Total records collected: 750
Fetching page 5... Total records collected: 1000
Fetching page 6... Total records collected: 1250
Fetching page 7... Total records collected: 1500
Fetching page 8... Total records collected: 1750
Fetching page 9... Total records collected: 2000
Fetching page 10... Total records collected: 2250
Fetching page 11... Total records collected: 2500
Fetching page 12... Total records collected: 2750
Fetching page 13... Total records collected: 3000
Fetching page 14... Total records collected: 3250
Fetching page 15... Total records collected: 3500
Fetching page 16... Total records collected: 3750
Fetching page 17... Total records collected: 4000
Fetching page 18... Total records collected: 4250
Fetching page 19... Total records collected: 4500
Fetching page 20... Total records collected: 4750
Fetching page 21... Total rec

In [7]:
#merge product names and quantities into dictionary
for order in orders:
    order.attributes['item_titles'] = [item.title for item in order.line_items]
    order.attributes['item_quantities'] = [item.quantity for item in order.line_items]

In [9]:
#flatten orders and make it a pandas dataframe
flattened_orders = []

for order in orders:
    # Start with the base order data
    base_info = order.attributes.copy()
    
    # Remove the complex objects to keep the dictionary clean
    base_info.pop('line_items', None)
    base_info.pop('customer', None)
    
    for item in order.line_items:
        # Create a new dictionary combining Order data + current Item data
        combined_row = {**base_info} # Copy order data
        combined_row['product_title'] = item.title
        combined_row['product_quantity'] = item.quantity
        combined_row['product_price'] = item.price

        flattened_orders.append(combined_row)

In [28]:
orders_df = pd.DataFrame(flattened_orders)

## Step 2: Cleaning

In [11]:
orders_df.head(10)

,id,admin_graphql_api_id,app_id,browser_ip,buyer_accepts_marketing,cancel_reason,cancelled_at,cart_token,checkout_id,checkout_token,...,fulfillments,payment_terms,refunds,shipping_address,shipping_lines,item_titles,item_quantities,product_title,product_quantity,product_price
0,1098588422190,gid://shopify/Order/1098588422190,457101,None,False,None,None,None,NaN,None,...,[fulfillment(998317260846)],None,[],shipping_address(None),[shipping_line(876670255150)],[Badfish Box],[1],Badfish Box,1,49.95
1,1098867114030,gid://shopify/Order/1098867114030,457101,None,False,None,None,None,NaN,None,...,[fulfillment(984002986030)],None,[],shipping_address(None),[shipping_line(876867878958)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
2,1098949460014,gid://shopify/Order/1098949460014,457101,None,False,None,None,None,NaN,None,...,[fulfillment(984003051566)],None,[],shipping_address(None),[shipping_line(876901302318)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
3,1099038064686,gid://shopify/Order/1099038064686,457101,None,False,None,None,None,NaN,None,...,[fulfillment(984003018798)],None,[],shipping_address(None),[shipping_line(876974080046)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
4,1099474042926,gid://shopify/Order/1099474042926,457101,None,False,None,None,None,NaN,None,...,[fulfillment(984003117102)],None,[],shipping_address(None),[shipping_line(877376864302)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
5,1099882659886,gid://shopify/Order/1099882659886,580111,174.202.59.115,True,None,None,,8.000757e+12,173be46654b6bf0d3dcb3ec63b808d02,...,[fulfillment(984003280942)],None,[],shipping_address(None),[shipping_line(877691174958)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
6,1100074713134,gid://shopify/Order/1100074713134,457101,None,False,None,None,None,NaN,None,...,[fulfillment(984003215406)],None,[],shipping_address(None),[shipping_line(877842038830)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
7,1100966199342,gid://shopify/Order/1100966199342,580111,72.69.100.104,True,None,None,47d06fd41c791d1c7e56e6c10de91377,8.000624e+12,2648a8deef64eb2faa611dfad2fcf777,...,[fulfillment(984003346478)],None,[],shipping_address(None),[shipping_line(878631125038)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
8,1101678706734,gid://shopify/Order/1101678706734,457101,None,False,None,None,None,NaN,None,...,[fulfillment(984003444782)],None,[],shipping_address(None),[shipping_line(879197945902)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95
9,1104810147886,gid://shopify/Order/1104810147886,457101,None,False,None,None,None,NaN,None,...,[fulfillment(984003510318)],None,[],shipping_address(None),[shipping_line(881769185326)],[Badfish Box - Northeast Inshore],[1],Badfish Box - Northeast Inshore,1,49.95


In [12]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39989 entries, 0 to 39988
Data columns (total 94 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   id                                          39989 non-null  int64  
 1   admin_graphql_api_id                        39989 non-null  object 
 2   app_id                                      39989 non-null  int64  
 3   browser_ip                                  20943 non-null  object 
 4   buyer_accepts_marketing                     39989 non-null  bool   
 5   cancel_reason                               42 non-null     object 
 6   cancelled_at                                42 non-null     object 
 7   cart_token                                  19736 non-null  object 
 8   checkout_id                                 22055 non-null  float64
 9   checkout_token                              22055 non-null  object 
 10  client_det

In [29]:
#Select desired columns
orders_df =  orders_df[['id','order_number','product_title','product_quantity','product_price','total_price','total_discounts',
                        'cancel_reason','cancelled_at','referring_site','landing_site','created_at','processed_at','closed_at','shipping_address', 
                        'buyer_accepts_marketing']]

In [24]:
orders_df.tail(5)

,id,order_number,product_title,product_quantity,product_price,total_price,total_discounts,cancel_reason,cancelled_at,referring_site,landing_site,created_at,processed_at,closed_at,shipping_address,buyer_accepts_marketing
39984,6431415140417,34781,Hat sleeve,1,1.00,0.00,1.00,None,None,None,None,2026-02-16 13:27:05+00:00,2026-02-16 13:27:04+00:00,2026-02-16 13:52:11+00:00,shipping_address(None),True
39985,6431417598017,34782,Resupply,1,1.00,0.00,1.00,None,None,None,None,2026-02-16 13:30:41+00:00,2026-02-16 13:30:41+00:00,2026-02-16 13:52:12+00:00,shipping_address(None),False
39986,6431422611521,34783,Hat Box,1,1.00,0.00,1.00,None,None,None,None,2026-02-16 13:36:26+00:00,2026-02-16 13:36:25+00:00,2026-02-16 13:37:51+00:00,shipping_address(None),True
39987,6431448531009,34784,Boatworks Tee,1,34.00,40.99,0.00,None,None,https://badfishsupply.com/,/collections/shirts?gad_source=1&gad_campaigni...,2026-02-16 14:04:37+00:00,2026-02-16 14:04:34+00:00,2026-02-16 14:13:38+00:00,shipping_address(None),False
39988,6431486902337,34785,Hat Sleeve,1,1.00,0.00,1.00,None,None,None,None,2026-02-16 14:34:29+00:00,2026-02-16 14:34:28+00:00,2026-02-16 14:34:58+00:00,shipping_address(None),False


In [30]:
#Change date columns to date time
date_columns = ['created_at','processed_at','closed_at']
for column in date_columns:
    orders_df[column] = pd.to_datetime(orders_df[column], utc=True)

In [31]:
#Convert price column to numeric
orders_df['total_price'] = pd.to_numeric(orders_df['total_price'], errors='coerce')

In [32]:
#Filter null values
orders_df = orders_df[
    (orders_df["total_price"] > 0) &
    (orders_df["cancelled_at"].isna()) &
    (orders_df["id"] != "NA") &
    (orders_df["shipping_address"] != "NA")
    ]

In [37]:
#Filter out all transactions made before 2023
orders_df = orders_df[orders_df['created_at'].dt.year >= 2023]

In [38]:
orders_df.to_csv('orders_23-pres.csv')